# Grokking sweep on a Colab GPU

Runs the weight-decay × train-fraction sweep from `grokking-progress-measures` on a GPU. Results go to Google Drive, and the sweep is **resumable**: if the session drops, rerun the sweep cell and it skips finished runs.

**Runtime → Change runtime type → T4 GPU**, then run all cells. At the end, download `grok_results.zip` (last cell) and put it in the repo's `results/` folder (or send it back to Claude).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/grokking-progress-measures/sweep'
!mkdir -p {OUT}

In [ ]:
!git clone -q https://github.com/kishorhange111/grokking-progress-measures.git /content/repo 2>/dev/null || git -C /content/repo pull -q
%cd /content/repo
!pip install -q -e .

## Sweep

6 configs (3 train fractions × 2 weight decays), 30k full-batch steps each, stopped 600 steps after reaching 99% test accuracy. Four processes share the GPU. A T4 should take roughly 10–30 minutes (an estimate, not measured).

In [ ]:
!python -m grok.cli sweep --p 59 --fracs 0.3 0.5 0.7 --wds 0.3 1.0 --seeds 0 1 2 \n    --steps 30000 --eval-every 200 --stop-after 600 \n    --workers 4 --device cuda --out {OUT}

## Long single run (the progress-measure figure)

The CPU pilot (frac 0.3, wd 1.0) memorised by step 200 but had not generalised by 25k steps. This runs it for 100k steps to see whether Fourier sparsity keeps rising ahead of test accuracy.

In [ ]:
!python -m grok.cli single --p 59 --frac 0.3 --wd 1.0 --seed 0 --steps 100000 --eval-every 250 --device cuda --out {OUT}/..

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open(f'{OUT}/sweep.md').read()))
display(Image(f'{OUT}/sweep.png'))

In [ ]:
!cd {OUT}/.. && zip -qr /content/grok_results.zip . && ls -la /content/grok_results.zip
from google.colab import files
files.download('/content/grok_results.zip')